# Kisan Mitra crop-photo classifier: lab photos vs field photos

Trains the photo classifier the backend uses, and measures the gap that matters: a model
trained on PlantVillage (leaves photographed on a plain background) tested on PlantDoc
(photos taken in fields and gardens).

- **A**: EfficientNet-B0 fine-tuned on PlantVillage only.
- **B**: the same, with PlantDoc's training photos mixed in.

Both are scored on the PlantVillage test split and on PlantDoc's test photos, after
removing test photos that are near-duplicates of a training photo. The confidence
threshold below which the app says "I'm not sure" is chosen on a validation split,
never on the test photos. Model B is exported to ONNX for the backend.

Run with a GPU runtime (Runtime > Change runtime type > T4 GPU). Takes about 30 minutes.

In [ ]:
import subprocess, sys, os, time
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm", "onnx", "onnxruntime", "imagehash"], check=True)
import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
T0 = time.time()

In [ ]:
# Data. PlantVillage colour images (spMohanty/PlantVillage-Dataset) and PlantDoc (pratikkayal/PlantDoc-Dataset).
import os, subprocess
if not os.path.isdir("/content/pv/raw/color"):
    subprocess.run("git clone -q --depth 1 --filter=blob:none --sparse https://github.com/spMohanty/PlantVillage-Dataset /content/pv", shell=True, check=True)
    subprocess.run("git -C /content/pv sparse-checkout set raw/color", shell=True, check=True)
if not os.path.isdir("/content/plantdoc/train"):
    subprocess.run("git clone -q --depth 1 https://github.com/pratikkayal/PlantDoc-Dataset /content/plantdoc", shell=True, check=True)
PV_ROOT = "/content/pv/raw/color"
PV_CLASSES = sorted(d for d in os.listdir(PV_ROOT) if os.path.isdir(os.path.join(PV_ROOT, d)))
PD_CLASSES = sorted(d for d in os.listdir("/content/plantdoc/train") if os.path.isdir(os.path.join("/content/plantdoc/train", d)))
print(len(PV_CLASSES), "PlantVillage classes")
print(len(PD_CLASSES), "PlantDoc classes:", PD_CLASSES)

In [ ]:
# PlantDoc folder -> PlantVillage class. Matched on lower-case names; anything unmatched is printed and skipped.
PD_TO_PV = {
    "apple scab leaf": "Apple___Apple_scab",
    "apple leaf": "Apple___healthy",
    "apple rust leaf": "Apple___Cedar_apple_rust",
    "bell_pepper leaf": "Pepper,_bell___healthy",
    "bell_pepper leaf spot": "Pepper,_bell___Bacterial_spot",
    "blueberry leaf": "Blueberry___healthy",
    "cherry leaf": "Cherry_(including_sour)___healthy",
    "corn gray leaf spot": "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot",
    "corn leaf blight": "Corn_(maize)___Northern_Leaf_Blight",
    "corn rust leaf": "Corn_(maize)___Common_rust_",
    "peach leaf": "Peach___healthy",
    "potato leaf": "Potato___healthy",
    "potato leaf early blight": "Potato___Early_blight",
    "potato leaf late blight": "Potato___Late_blight",
    "raspberry leaf": "Raspberry___healthy",
    "soyabean leaf": "Soybean___healthy",
    "soybean leaf": "Soybean___healthy",
    "squash powdery mildew leaf": "Squash___Powdery_mildew",
    "strawberry leaf": "Strawberry___healthy",
    "tomato early blight leaf": "Tomato___Early_blight",
    "tomato septoria leaf spot": "Tomato___Septoria_leaf_spot",
    "tomato leaf": "Tomato___healthy",
    "tomato leaf bacterial spot": "Tomato___Bacterial_spot",
    "tomato leaf late blight": "Tomato___Late_blight",
    "tomato leaf mosaic virus": "Tomato___Tomato_mosaic_virus",
    "tomato leaf yellow virus": "Tomato___Tomato_Yellow_Leaf_Curl_Virus",
    "tomato mold leaf": "Tomato___Leaf_Mold",
    "tomato two spotted spider mites leaf": "Tomato___Spider_mites Two-spotted_spider_mite",
    "grape leaf": "Grape___healthy",
    "grape leaf black rot": "Grape___Black_rot",
}
missing_pv = sorted(set(PD_TO_PV.values()) - set(PV_CLASSES))
print("mapping targets not in PlantVillage:", missing_pv)
CLS_IDX = {c: i for i, c in enumerate(PV_CLASSES)}

def pd_items(split):
    root = f"/content/plantdoc/{split}"
    items, skipped = [], {}
    for d in sorted(os.listdir(root)):
        p = os.path.join(root, d)
        if not os.path.isdir(p):
            continue
        target = PD_TO_PV.get(d.strip().lower())
        files = [os.path.join(p, f) for f in sorted(os.listdir(p)) if f.lower().endswith((".jpg", ".jpeg", ".png"))]
        if target is None or target not in CLS_IDX:
            skipped[d] = len(files)
            continue
        items += [(f, CLS_IDX[target]) for f in files]
    return items, skipped

pd_train_all, sk1 = pd_items("train")
pd_test_all, sk2 = pd_items("test")
print("PlantDoc train", len(pd_train_all), "test", len(pd_test_all), "skipped", sk1, sk2)

In [ ]:
# Shrink PlantDoc photos once (some are several thousand pixels wide), dropping files that don't open.
from PIL import Image
from concurrent.futures import ThreadPoolExecutor
os.makedirs("/content/pd_small", exist_ok=True)

def shrink(item):
    src, y = item
    dst = "/content/pd_small/" + src.replace("/content/plantdoc/", "").replace("/", "__")
    dst = os.path.splitext(dst)[0] + ".jpg"
    if not os.path.exists(dst):
        try:
            im = Image.open(src).convert("RGB")
            im.thumbnail((448, 448))
            im.save(dst, quality=92)
        except Exception:
            return None
    return (dst, y)

with ThreadPoolExecutor(8) as ex:
    pd_train_all = [r for r in ex.map(shrink, pd_train_all) if r]
    pd_test_all = [r for r in ex.map(shrink, pd_test_all) if r]
print("readable PlantDoc train", len(pd_train_all), "test", len(pd_test_all))

# Near-duplicate check: drop test photos whose perceptual hash is within 6 bits of any training photo.
import imagehash, numpy as np
def phash(p):
    return imagehash.phash(Image.open(p)).hash.flatten()
with ThreadPoolExecutor(8) as ex:
    H_tr = np.array(list(ex.map(lambda t: phash(t[0]), pd_train_all)))
    H_te = np.array(list(ex.map(lambda t: phash(t[0]), pd_test_all)))
dist = (H_te[:, None, :] != H_tr[None, :, :]).sum(-1).min(1)
pd_test = [t for t, d in zip(pd_test_all, dist) if d > 6]
n_dupes = len(pd_test_all) - len(pd_test)
print("PlantDoc test photos removed as near-duplicates of training photos:", n_dupes, "kept", len(pd_test))

In [ ]:
# Splits. PlantVillage: stratified 80/10/10 by image (it has no leaf or plant ids, so photos of the same
# leaf may sit on both sides; its test score is an upper bound). PlantDoc train: 85% train, 15% validation.
import random
from collections import Counter
rng = random.Random(7)
pv_items = []
for c in PV_CLASSES:
    fs = sorted(os.listdir(os.path.join(PV_ROOT, c)))
    pv_items += [(os.path.join(PV_ROOT, c, f), CLS_IDX[c]) for f in fs if f.lower().endswith((".jpg", ".jpeg", ".png"))]
by_cls = {}
for it in pv_items:
    by_cls.setdefault(it[1], []).append(it)
pv_train, pv_val, pv_test = [], [], []
for y, its in by_cls.items():
    rng.shuffle(its)
    n = len(its); a, b = int(n * 0.8), int(n * 0.9)
    pv_train += its[:a]; pv_val += its[a:b]; pv_test += its[b:]
by_cls = {}
for it in pd_train_all:
    by_cls.setdefault(it[1], []).append(it)
pd_train, pd_val = [], []
for y, its in by_cls.items():
    rng.shuffle(its)
    k = max(1, int(round(len(its) * 0.15)))
    pd_val += its[:k]; pd_train += its[k:]
print("PlantVillage", len(pv_train), len(pv_val), len(pv_test), "| PlantDoc train", len(pd_train), "val", len(pd_val), "test", len(pd_test))

In [ ]:
import torch, timm, torch.nn as nn, numpy as np
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms as T
DEV = "cuda" if torch.cuda.is_available() else "cpu"
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
train_tf = T.Compose([T.RandomResizedCrop(224, scale=(0.35, 1.0)), T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
                      T.RandomRotation(20), T.ColorJitter(0.35, 0.35, 0.35, 0.05), T.ToTensor(), T.Normalize(MEAN, STD)])
eval_tf = T.Compose([T.Resize(256), T.CenterCrop(224), T.ToTensor(), T.Normalize(MEAN, STD)])

class DS(Dataset):
    def __init__(self, items, tf): self.items, self.tf = items, tf
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        p, y = self.items[i]
        return self.tf(Image.open(p).convert("RGB")), y

def loader(items, tf, shuffle=False, sampler=None, bs=64):
    return DataLoader(DS(items, tf), batch_size=bs, shuffle=shuffle and sampler is None, sampler=sampler,
                      num_workers=os.cpu_count(), pin_memory=True, persistent_workers=False)

@torch.no_grad()
def predict(model, items):
    model.eval(); out = []
    for x, _ in loader(items, eval_tf, bs=128):
        with torch.autocast(DEV, dtype=torch.float16, enabled=DEV == "cuda"):
            out.append(torch.softmax(model(x.to(DEV, non_blocking=True)).float(), 1).cpu())
    return torch.cat(out).numpy()

def train(items_main, items_extra=None, epochs=4, seed=0):
    torch.manual_seed(seed)
    model = timm.create_model("efficientnet_b0", pretrained=True, num_classes=len(PV_CLASSES)).to(DEV)
    items = items_main + (items_extra or [])
    if items_extra:
        # Field photos are about 5% of the data; sample them so they are about 30% of each epoch.
        w = [0.7 / len(items_main)] * len(items_main) + [0.3 / len(items_extra)] * len(items_extra)
        sampler = WeightedRandomSampler(w, num_samples=len(items_main), replacement=True)
    else:
        sampler = None
    dl = loader(items, train_tf, shuffle=True, sampler=sampler)
    opt = torch.optim.AdamW(model.parameters(), lr=4e-4, weight_decay=0.02)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=4e-4, total_steps=epochs * len(dl), pct_start=0.15)
    scaler = torch.cuda.amp.GradScaler(enabled=DEV == "cuda")
    lossf = nn.CrossEntropyLoss(label_smoothing=0.1)
    for ep in range(epochs):
        model.train(); t = time.time(); tot = 0
        for x, y in dl:
            x, y = x.to(DEV, non_blocking=True), y.to(DEV, non_blocking=True)
            with torch.autocast(DEV, dtype=torch.float16, enabled=DEV == "cuda"):
                loss = lossf(model(x), y)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            tot += loss.item()
        print(f"epoch {ep+1}/{epochs} loss {tot/len(dl):.3f} {time.time()-t:.0f}s", flush=True)
    return model

In [ ]:
from sklearn.metrics import f1_score, accuracy_score

def crop_of(c): return c.split("___")[0]
def healthy(c): return c.endswith("healthy")

def scores(P, items):
    y = np.array([t[1] for t in items]); yhat = P.argmax(1)
    present = sorted(set(y.tolist()))
    top3 = np.mean([yy in np.argsort(-p)[:3] for p, yy in zip(P, y)])
    crop_acc = np.mean([crop_of(PV_CLASSES[a]) == crop_of(PV_CLASSES[b]) for a, b in zip(yhat, y)])
    hd = np.mean([healthy(PV_CLASSES[a]) == healthy(PV_CLASSES[b]) for a, b in zip(yhat, y)])
    rec = {PV_CLASSES[c]: round(float(np.mean(yhat[y == c] == c)), 3) for c in present}
    return {"n": int(len(y)), "accuracy": round(float(accuracy_score(y, yhat)), 4),
            "macro_f1": round(float(f1_score(y, yhat, labels=present, average="macro", zero_division=0)), 4),
            "top3": round(float(top3), 4), "crop_accuracy": round(float(crop_acc), 4),
            "healthy_vs_diseased_accuracy": round(float(hd), 4), "recall_per_class": rec}

def abstain_curve(P, items):
    y = np.array([t[1] for t in items]); conf = P.max(1); ok = P.argmax(1) == y
    rows = []
    for th in np.round(np.arange(0.0, 0.96, 0.05), 2):
        m = conf >= th
        rows.append({"threshold": float(th), "coverage": round(float(m.mean()), 4),
                     "accuracy_when_answered": round(float(ok[m].mean()) if m.any() else 0.0, 4)})
    return rows

def pick_threshold(curve, target=0.85):
    # Lowest threshold whose answered accuracy on the validation split reaches the target.
    for r in curve:
        if r["accuracy_when_answered"] >= target and r["coverage"] > 0:
            return r["threshold"]
    return curve[-1]["threshold"]

def at_threshold(P, items, th):
    y = np.array([t[1] for t in items]); conf = P.max(1); ok = P.argmax(1) == y; m = conf >= th
    return {"threshold": th, "coverage": round(float(m.mean()), 4),
            "accuracy_when_answered": round(float(ok[m].mean()) if m.any() else 0.0, 4)}

In [ ]:
RESULTS = {"data": {"pv_train": len(pv_train), "pv_val": len(pv_val), "pv_test": len(pv_test),
                    "pd_train": len(pd_train), "pd_val": len(pd_val), "pd_test": len(pd_test),
                    "pd_test_near_duplicates_removed": int(n_dupes), "n_classes": len(PV_CLASSES),
                    "pd_classes_mapped": sorted(set(PD_TO_PV.values()) & set(PV_CLASSES)).__len__()}}
models = {}
for name, extra in [("A_plantvillage_only", None), ("B_plus_plantdoc", pd_train)]:
    print("training", name, flush=True)
    t = time.time()
    m = train(pv_train, extra)
    models[name] = m
    Pv, Pd, Pval = predict(m, pv_test), predict(m, pd_test), predict(m, pd_val)
    curve = abstain_curve(Pval, pd_val); th = pick_threshold(curve)
    RESULTS[name] = {"train_seconds": round(time.time() - t), "plantvillage_test": scores(Pv, pv_test),
                     "plantdoc_test": scores(Pd, pd_test), "field_val_curve": curve,
                     "threshold_from_val": th, "plantdoc_test_at_threshold": at_threshold(Pd, pd_test, th),
                     "plantdoc_test_curve": abstain_curve(Pd, pd_test)}
    r = RESULTS[name]
    print(name, "PV acc", r["plantvillage_test"]["accuracy"], "| PlantDoc acc", r["plantdoc_test"]["accuracy"],
          "macroF1", r["plantdoc_test"]["macro_f1"], "| at th", th, r["plantdoc_test_at_threshold"], flush=True)

In [ ]:
# Export model B to ONNX, check it against PyTorch using the exact numpy preprocessing the backend uses,
# and time it on this runtime's CPU.
import onnxruntime as ort, hashlib, json

def preprocess_np(path):
    im = Image.open(path).convert("RGB")
    w, h = im.size
    if w <= h:
        nw, nh = 256, int(256 * h / w)
    else:
        nw, nh = int(256 * w / h), 256
    im = im.resize((nw, nh), Image.BILINEAR)
    left, top = int(round((nw - 224) / 2.0)), int(round((nh - 224) / 2.0))
    im = im.crop((left, top, left + 224, top + 224))
    x = (np.asarray(im, dtype=np.float32) / 255.0 - np.array(MEAN, np.float32)) / np.array(STD, np.float32)
    return x.transpose(2, 0, 1)[None]

mB = models["B_plus_plantdoc"].eval().float().cpu()
os.makedirs("/content/export", exist_ok=True)
onnx_path = "/content/export/crop_disease_effb0.onnx"
torch.onnx.export(mB, torch.randn(1, 3, 224, 224), onnx_path, input_names=["image"], output_names=["logits"],
                  dynamic_axes={"image": {0: "batch"}, "logits": {0: "batch"}}, opset_version=17)
sess = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
diffs, agree = [], 0
for p, _ in pd_test[:40]:
    x = preprocess_np(p)
    with torch.no_grad():
        ref = torch.softmax(mB(eval_tf(Image.open(p).convert("RGB"))[None]), 1).numpy()
    lo = sess.run(None, {"image": x})[0]; got = np.exp(lo - lo.max()); got /= got.sum()
    diffs.append(float(np.abs(ref - got).max())); agree += int(ref.argmax() == got.argmax())
x = preprocess_np(pd_test[0][0]); lat = []
for i in range(60):
    t = time.perf_counter(); sess.run(None, {"image": x}); lat.append((time.perf_counter() - t) * 1000)
lat = np.array(lat[10:])
sha = hashlib.sha256(open(onnx_path, "rb").read()).hexdigest()
thB = RESULTS["B_plus_plantdoc"]["threshold_from_val"]
labels = {"model": "efficientnet_b0 fine-tuned on PlantVillage + PlantDoc train", "classes": PV_CLASSES,
          "threshold": thB, "input": {"size": 224, "resize_short_side": 256, "mean": MEAN, "std": STD,
          "layout": "NCHW RGB float32"}, "sha256": sha}
json.dump(labels, open("/content/export/crop_disease_labels.json", "w"), indent=1)
RESULTS["onnx"] = {"bytes": os.path.getsize(onnx_path), "sha256": sha, "max_prob_diff_vs_torch": round(max(diffs), 6),
                   "argmax_agreement": f"{agree}/40", "cpu_ms_p50": round(float(np.percentile(lat, 50)), 1),
                   "cpu_ms_p95": round(float(np.percentile(lat, 95)), 1), "cpu": os.cpu_count()}
RESULTS["minutes_total"] = round((time.time() - T0) / 60, 1)
print(RESULTS["onnx"])

In [ ]:
import json, shutil
json.dump(RESULTS, open("/content/export/classifier_results.json", "w"), indent=1)
shutil.make_archive("/content/kisan_mitra_classifier", "zip", "/content/export")
print("===KM_RESULTS_BEGIN===")
print(json.dumps(RESULTS, separators=(",", ":")))
print("===KM_RESULTS_END===")